In [1]:
import re, json, time, hashlib, requests
import pandas as pd

from fastai.vision.all import *
from pathlib import Path
from fastdownload import download_url

In [6]:
HEADERS = {"User-Agent": "personal-learning-project"}
CACHE = Path("cache")  # Saved API responses, so re-running doesn't re-download everything
COVERS = Path("covers")  # covers/{era}/{cover_id}.jpg - genres live in books.csv, not in folders
PROBE = Path("probe")  # Classics
# In ML, a "probe set" means examples you run through a model to poke at its behaviour, as opposed to training or validating it. 

In [ ]:
ERAS = {
    "2000_2004": (2000, 2004),
    "2010_2014": (2010, 2014),
    "2021_2025": (2021, 2025),
}

In [4]:
GENRES = {
    "romance":  ["romance", "love stories"],
    "fantasy":  ["fantasy", "magic"],
    "thriller": ["thriller", "suspense", "mystery", "crime"],
    "scifi":    ["science fiction"],
    "horror":   ["horror"],
}

In [ ]:
N_PER_GENRE_PER_ERA = 100     # same in every era to keep it fair
MAX_PER_AUTHOR = 2  # To avoid the model learning by author name

In [2]:
def search_books(genre, start_year, end_year, page=1):
    """Return one page (up to 100) of books matching any of this genre's keywords, first published in the year range.
    Each book is a dict with key, title, author_name, first_publish_year, subject."""
    # we search by "science fiction" rather than "sci-fi"
    keywords = " OR ".join(f'"{g}"' for g in GENRES[genre])  # we need the quotes as subject: science fiction and 
    # subject:"science fiction" return different results with the former also matching science as a subject and 
    # matching fiction against any field
    params = {
        "q": f"subject: ({keywords}) AND first_publish_year: [{start_year} TO {end_year}]",
        "fields": "key,title,author_name,first_publish_year,subject",
        "limit": 100,
        "page": page,
    }

    # https://openlibrary.org/search.json?q=subject%3a%22romance%22+AND+first_publish_year%3a%5b2021+TO+2025%5d&limit=100&page=1
    
    # fields controls what comes back in the response - we choose the columns we want otherwise we get a lot of unnecessary
    # fields which makes the response bigger and slower
    
    # Lets say there are 50,000 matching books, the server won't send them all at once, so it splits them into pages
    # where there are 100 books per page (limit=100) and page=1 gives books 1-100, page=2 gives books 101-200 and so on
    # This is why the main loop does page += 1 until we have enough books
    # Saved to CACHE so re-running doesn't ask the API again. The short hash of the query is in the
    # file name so that changing the keywords in GENRES gives a new file instead of reusing old results.
    query_hash = hashlib.md5(params["q"].encode()).hexdigest()[:8]
    cache_file = CACHE / "search" / f"{genre}_{start_year}_{end_year}_p{page}_{query_hash}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text())["docs"]

    r = requests.get("https://openlibrary.org/search.json", 
                    params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()  # stop on errors (e.g. 429) so a failed response never gets cached

    time.sleep(1)
    # Open Library is a free service run by a non-profit (the Internet Archive). Firing hundreds of requests per second can
    # overload it, and it may block you temporarily (HTTP 429 "Too Many Requests"). Waiting one second between requests keeps you safely under that.
    
    data = r.json()
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    cache_file.write_text(json.dumps(data))
    return data["docs"]

In [7]:
# CHECK
docs = search_books("romance", 2021, 2025)
len(docs), docs[0]

(100,
 {'author_name': ['Joana Marcus'],
  'first_publish_year': 2021,
  'key': '/works/OL26128903W',
  'title': 'Before December',
  'subject': ['Romance',
   'Christmas romance',
   'second chance romance',
   'forbidden love',
   'family',
   'friendship',
   'love triangle',
   'secrets',
   'lies',
   'forgiveness',
   'redemption',
   'love story',
   'holiday romance',
   'small town romance']})

In [22]:
def subjects_to_genres(subjects):
    """['Fiction, romance, contemporary', 'Magic'] -> {'romance', 'fantasy'}"""
    found = set()
    for subject in subjects:
        for genre, keywords in GENRES.items():
            for keyword in keywords:              # check each keyword separately
                if keyword in subject.lower():
                    found.add(genre)

    return found

In [23]:
# CHECK — should print {'romance', 'fantasy'}
subjects_to_genres(["Fiction, romance, contemporary", "Magic"])

{'fantasy', 'romance'}

## Picking the cover edition

Open Library stores books on two levels:
- A **work** is the book as a story, e.g. *Twisted Love* by Ana Huang, whatever version.
- An **edition** is one specific version of that work: the 2021 US paperback, the 2022 UK hardback, the German translation, the audiobook...
  **Each edition has its own cover, publication date, language and ISBN.**

The search only gives us *works*, and a work doesn't say which edition its cover belongs to or when that edition was printed.
So for every book we make a second request to `/works/OL…W/editions.json` ("editions call") to see all its editions,
and pick the cover of an **English edition printed when the book first came out** (or up to a year later),
so the cover belongs to the same era as the book.

That's one editions call per book we check, so they are most of the requests in this notebook - which is why they are cached.

In [ ]:
def get_editions(work_key, offset=0, limit=100):
    """Return the editions JSON for a work. Saved to CACHE so re-running the notebook doesn't ask the API again."""
    # Caching: every response is saved as a JSON file named after the work (and which page of editions it is).
    # Before asking the API we check if that file already exists - if it does we read it from disk instead,
    # which is instant and sends no request. Delete the cache folder if you ever want fresh data.
    work_id = work_key.split("/")[-1]  # "/works/OL123W" -> "OL123W"
    cache_file = CACHE / f"{work_id}_{offset}_{limit}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text())

    # Not cached yet -> ask Open Library for this work's editions.
    # Most books have fewer than 100 editions so one page (limit=100) is enough; classics have thousands,
    # so for those we ask for more pages using offset (editions 0-999, 1000-1999, ...).
    r = requests.get(f"https://openlibrary.org{work_key}/editions.json",
                     params={"limit": limit, "offset": offset}, headers=HEADERS, timeout=30)
    r.raise_for_status()  # stop on errors (e.g. 429) so a failed response never gets cached
    time.sleep(1)
    data = r.json()  # {"size": total number of editions, "entries": [one dict per edition], "links": {...}}

    # Save it for next time
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    cache_file.write_text(json.dumps(data))
    return data

In [ ]:
def extract_year(text):
    """'Mar 01, 2005' -> 2005, 'March 2005' -> 2005, None/junk -> None"""
    match = re.search(r"\d{4}", text or "")
    return int(match.group()) if match else None


def is_english(edition):
    """Use the language field if there is one, otherwise guess from the ISBN.
    ISBN-13s starting 9780, 9781 or 9798 are English-language publishers."""
    langs = [l["key"] for l in edition.get("languages", [])]
    if langs:
        return "/languages/eng" in langs
    isbns = edition.get("isbn_13", [])
    return any(isbn.startswith(("9780", "9781", "9798")) for isbn in isbns)


def cover_of(edition):
    """First real cover id of an edition, or None (-1 means 'no cover')."""
    covers = [c for c in edition.get("covers", []) if c > 0]
    return covers[0] if covers else None

In [ ]:
def pick_first_edition(work_key, first_year, era_end):
    """Return (cover_id, edition_year) for the earliest English edition with a cover,
    printed in first_year or the year after (but still inside the era). (None, None) if there isn't one."""
    # We allow one extra year because the very first edition sometimes has no cover on Open Library,
    # (e.g. "Hopeless", 2013) but the edition after it does - and its cover is still from the same era.
    last_year = min(first_year + 1, era_end)
    best = (None, None)  # nothing found yet

    # Go through every edition of the book and keep the earliest one that passes all the checks
    for ed in get_editions(work_key).get("entries", []):
        ed_year = extract_year(ed.get("publish_date"))  # publish_date is free text like "Mar 01, 2005"
        cover_id = cover_of(ed)
        if ed_year is None or not (first_year <= ed_year <= last_year):  # a later reprint -> different cover era
            continue
        if not is_english(ed) or cover_id is None:  # translation, or no cover uploaded
            continue
        if best[1] is None or ed_year < best[1]:  # keep the earliest one
            best = (cover_id, ed_year)

    return best

In [ ]:
# CHECK - a cover id and a year, or (None, None) if this book has no usable first edition
pick_first_edition(docs[0]["key"], docs[0]["first_publish_year"], 2025)

## Collecting the books

For every era and genre we keep going through search pages until we have `N_PER_GENRE_PER_ERA` books **that have a usable cover**.

⚠️ Every book needs one editions request (~1 second each), so the full run with `N_PER_GENRE_PER_ERA = 100` takes **about an hour**.
Try it with `N_PER_GENRE_PER_ERA = 5` first.

All requests (searches and editions) are cached in `CACHE`, so if the loop stops halfway or you run it again,
everything already fetched is read from disk instantly and only new books cause requests.

In [ ]:
books = []
seen = set()  # (title, author) pairs we already have
author_count = {}  # author -> how many of their books we have

for era, (start_year, end_year) in ERAS.items():
    for genre in GENRES:
        count = 0
        page = 1
        while count < N_PER_GENRE_PER_ERA:
            # - page through search_books until you have N_PER_GENRE_PER_ERA books
            docs = search_books(genre, start_year, end_year, page)

            if len(docs) == 0:  # no more results
                break

            for doc in docs:
                if count >= N_PER_GENRE_PER_ERA:
                    break

                title  = doc.get("title", "")
                author = doc.get("author_name", ["unknown"])[0]
                genres = subjects_to_genres(doc.get("subject", []))
                book_id = (title.lower(), author.lower())

                # A book with several genres (e.g. romance + fantasy) is returned by more than one genre's search.
                # We keep it only the first time (in the genre that was searched first) - its all_genres column
                # still lists every genre, so no information is lost. Later searches skip it, and that genre
                # fills its N_PER_GENRE_PER_ERA with other books instead.
                if book_id in seen:
                    continue
                if author_count.get(author, 0) >= MAX_PER_AUTHOR:  # don't let one author dominate
                    continue
                if genre not in genres:  # our keywords don't agree with the search - skip to keep labels clean
                    continue

                # Editions call: find the cover of the first English edition (cached, see get_editions).
                # This comes after the cheap checks above so we don't make requests for books we'd skip anyway.
                cover_id, edition_year = pick_first_edition(doc["key"], doc["first_publish_year"], end_year)
                if cover_id is None:  # no English first edition with a cover
                    continue

                books.append(dict(
                    work_key     = doc["key"],
                    title        = title,
                    author       = author,
                    year         = doc["first_publish_year"],
                    edition_year = edition_year,
                    cover_id     = cover_id,
                    era          = era,
                    searched     = genre,
                    all_genres   = " ".join(sorted(genres)),
                    n_genres     = len(genres),
                ))

                seen.add(book_id)
                author_count[author] = author_count.get(author, 0) + 1
                count += 1

            page += 1
        print(era, genre, count)

In [ ]:
df = pd.DataFrame(books)
df["filename"] = df.era + "/" + df.cover_id.astype(str) + ".jpg"
df.to_csv("books.csv", index=False)  # save so you never have to redo the collection

df.groupby(["era", "searched"]).size()

From now on you can skip the collection and just load the table:

In [ ]:
df = pd.read_csv("books.csv")
df.head()

## Downloading the covers

In [ ]:
for _, row in df.iterrows():
    dest = COVERS / row.filename
    if dest.exists():
        continue  # already downloaded
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://covers.openlibrary.org/b/id/{row.cover_id}-L.jpg?default=false"  # default=false -> error instead of a blank image
    try:
        download_url(url, dest, show_progress=False)
    except Exception:
        pass  # missing cover - skip it

In [ ]:
# Same cleanup as the bird notebook: delete images that can't be opened
failed = verify_images(get_image_files(COVERS))
failed.map(Path.unlink)
len(failed)

In [ ]:
def looks_like_a_cover(f):
    """Drop tiny images (blurry -> the model could learn 'old = low quality') and odd shapes.
    Square images are usually audiobook covers, not printed books."""
    w, h = Image.open(f).size
    return h >= 250 and 0.55 <= w / h <= 0.75

df = df[df.filename.apply(lambda f: (COVERS / f).exists())]  # keep only rows whose image downloaded
df = df[df.filename.apply(lambda f: looks_like_a_cover(COVERS / f))]
df.to_csv("books_clean.csv", index=False)

print(len(df), "covers left")
df.groupby(["era", "searched"]).size()  # are the eras still balanced?

In [ ]:
# Share of books with more than one of our genres, per era - more blending in 2021_2025?
df.groupby("era").n_genres.apply(lambda s: (s > 1).mean())

## v1: one genre per book

Same as the bird notebook, except the images and labels come from the table:
`get_x` says which column has the image path, `get_y` which column has the answer.
v1 only uses books with exactly one genre.

Start with one era to check everything works:

In [ ]:
era = "2021_2025"
df_era = df[(df.era == era) & (df.n_genres == 1)]  # v1: single-genre books only

dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_x=ColReader("filename", pref=COVERS),  # which column has the image
    get_y=ColReader("all_genres"),  # which column has the answer
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms=[Resize(192, method="squish")],
).dataloaders(df_era, bs=32)  # a table, not a folder

dls.show_batch(max_n=9)  # do the labels match the covers?

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

Now every era. Each era is trained a few times with different validation splits (`seed`),
because with a few hundred images the score jumps around - a difference between eras only counts if it's bigger than that jumping around.

In [ ]:
SEEDS = [0, 1, 2]
results_v1 = []
learners_v1 = {}  # one model per era, used for the classics later

for era in ERAS:
    df_era = df[(df.era == era) & (df.n_genres == 1)]
    for seed in SEEDS:
        dls = DataBlock(
            blocks=(ImageBlock, CategoryBlock),
            get_x=ColReader("filename", pref=COVERS),
            get_y=ColReader("all_genres"),
            splitter=RandomSplitter(valid_pct=0.2, seed=seed),
            item_tfms=[Resize(192, method="squish")],
        ).dataloaders(df_era, bs=32)

        learn = vision_learner(dls, resnet18, metrics=error_rate)
        learn.fine_tune(3)
        loss, err = learn.validate()
        results_v1.append(dict(era=era, seed=seed, n_books=len(df_era), error_rate=err))

    learners_v1[era] = learn
    learn.export(f"model_v1_{era}.pkl")

# Lower error_rate = genres were easier to tell apart from the cover in that era
pd.DataFrame(results_v1).groupby("era").agg(n_books=("n_books", "first"),
                                            mean_error=("error_rate", "mean"),
                                            std_error=("error_rate", "std"))

## v2: several genres per book (Lesson 6 / book chapter 6)

Three changes from v1: keep **all** books, `MultiCategoryBlock` with `label_delim=" "` (so `"fantasy romance"` becomes two labels),
and macro F1 as the score (**higher** = better). `accuracy_multi` looks great even for a useless model, because most answers are "no".

In [ ]:
results_v2 = []

for era in ERAS:
    df_era = df[df.era == era]  # 1. all books, including multi-genre ones
    for seed in SEEDS:
        dls = DataBlock(
            blocks=(ImageBlock, MultiCategoryBlock),  # 2. several answers allowed
            get_x=ColReader("filename", pref=COVERS),
            get_y=ColReader("all_genres", label_delim=" "),  # "fantasy romance" -> ["fantasy", "romance"]
            splitter=RandomSplitter(valid_pct=0.2, seed=seed),
            item_tfms=[Resize(192, method="squish")],
        ).dataloaders(df_era, bs=32)

        learn = vision_learner(dls, resnet18, metrics=F1ScoreMulti(average="macro"))  # 3. different score
        learn.fine_tune(3)
        loss, f1 = learn.validate()
        results_v2.append(dict(era=era, seed=seed, n_books=len(df_era), f1=f1))

    learn.export(f"model_v2_{era}.pkl")

# Higher f1 = genres were easier to tell apart from the cover in that era
pd.DataFrame(results_v2).groupby("era").agg(n_books=("n_books", "first"),
                                            mean_f1=("f1", "mean"),
                                            std_f1=("f1", "std"))

## Classics: are old books being dressed up as other genres?

The classics are **never trained on**. We show their reprint covers from each era to that era's v1 model and ask
"what genre does this cover look like?". If the answer isn't one of the book's real genres, the cover is selling something else.

The true genres are a judgement call - change them if you disagree.

In [ ]:
CLASSICS = {  # title -> (author, true genres)
    "Frankenstein":                  ("Mary Shelley",        ["horror", "scifi"]),
    "Dracula":                       ("Bram Stoker",         ["horror"]),
    "The Strange Case of Dr. Jekyll and Mr. Hyde": ("Robert Louis Stevenson", ["horror"]),
    "The Picture of Dorian Gray":    ("Oscar Wilde",         ["horror", "fantasy"]),
    "The Turn of the Screw":         ("Henry James",         ["horror"]),
    "Pride and Prejudice":           ("Jane Austen",         ["romance"]),
    "Persuasion":                    ("Jane Austen",         ["romance"]),
    "Jane Eyre":                     ("Charlotte Brontë",    ["romance"]),
    "Wuthering Heights":             ("Emily Brontë",        ["romance"]),
    "The Time Machine":              ("H. G. Wells",         ["scifi"]),
    "The War of the Worlds":         ("H. G. Wells",         ["scifi"]),
    "Twenty Thousand Leagues Under the Sea": ("Jules Verne", ["scifi"]),
    "Brave New World":               ("Aldous Huxley",       ["scifi"]),
    "Nineteen Eighty-Four":          ("George Orwell",       ["scifi"]),
    "The Hound of the Baskervilles": ("Arthur Conan Doyle",  ["thriller"]),
    "The Moonstone":                 ("Wilkie Collins",      ["thriller"]),
    "Murder on the Orient Express":  ("Agatha Christie",     ["thriller"]),
    "Rebecca":                       ("Daphne du Maurier",   ["thriller", "romance"]),
    "Alice's Adventures in Wonderland": ("Lewis Carroll",    ["fantasy"]),
    "The Hobbit":                    ("J. R. R. Tolkien",    ["fantasy"]),
}
MAX_COVERS_PER_ERA = 5  # per classic, so one very-reprinted book doesn't dominate

In [ ]:
def find_work(title, author):
    """Return the work key of the most-printed work matching title + author (the 'real' one, not a study guide)."""
    # Cached like the other requests. File name from title + author, e.g. "dracula_bram_stoker.json"
    # (\W+ = anything that isn't a letter/number, replaced with "_" so it's a safe file name)
    cache_file = CACHE / "works" / (re.sub(r"\W+", "_", f"{title}_{author}").lower() + ".json")
    if cache_file.exists():
        docs = json.loads(cache_file.read_text())["docs"]
        return docs[0]["key"] if docs else None

    params = {"title": title, "author": author, "fields": "key,title", "sort": "editions", "limit": 1}
    r = requests.get("https://openlibrary.org/search.json", params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    time.sleep(1)
    data = r.json()
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    cache_file.write_text(json.dumps(data))
    docs = data["docs"]
    return docs[0]["key"] if docs else None


def classic_covers_in_era(work_key, start_year, end_year):
    """Up to MAX_COVERS_PER_ERA (cover_id, edition_year) pairs of English editions printed in the era.
    Classics have thousands of editions, so we go through them 1000 at a time."""
    found = []
    offset = 0
    while True:
        data = get_editions(work_key, offset=offset, limit=1000)
        for ed in data.get("entries", []):
            ed_year = extract_year(ed.get("publish_date"))
            cover_id = cover_of(ed)
            if ed_year and start_year <= ed_year <= end_year and is_english(ed) and cover_id:
                found.append((cover_id, ed_year))
        if "next" not in data.get("links", {}):  # no more pages of editions
            break
        offset += 1000
    found = list(dict.fromkeys(found))  # remove duplicate covers, keep order
    return found[:MAX_COVERS_PER_ERA]

In [ ]:
classic_rows = []
for title, (author, true_genres) in CLASSICS.items():
    work_key = find_work(title, author)
    if work_key is None:
        print("not found:", title)
        continue
    for era, (start_year, end_year) in ERAS.items():
        for cover_id, ed_year in classic_covers_in_era(work_key, start_year, end_year):
            classic_rows.append(dict(title=title, era=era, edition_year=ed_year, cover_id=cover_id,
                                     true_genres=" ".join(true_genres),
                                     filename=f"{era}/{cover_id}.jpg"))

classics = pd.DataFrame(classic_rows)
classics.groupby(["title", "era"]).size().unstack(fill_value=0)  # how many covers per classic per era

In [ ]:
for _, row in classics.iterrows():
    dest = PROBE / row.filename
    if dest.exists():
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://covers.openlibrary.org/b/id/{row.cover_id}-L.jpg?default=false"
    try:
        download_url(url, dest, show_progress=False)
    except Exception:
        pass

failed = verify_images(get_image_files(PROBE))
failed.map(Path.unlink)
classics = classics[classics.filename.apply(lambda f: (PROBE / f).exists())]
classics.to_csv("classics.csv", index=False)
len(classics)

In [ ]:
# The same classic across the eras
title = "Pride and Prejudice"
rows = classics[classics.title == title]
show_images([PILImage.create(PROBE / f) for f in rows.filename],
            titles=[str(y) for y in rows.edition_year], nrows=1, figsize=(3 * len(rows), 4))

In [ ]:
# Ask each era's v1 model what genre the classic's cover looks like
# (if you restarted the notebook: learners_v1 = {era: load_learner(f"model_v1_{era}.pkl") for era in ERAS})
predictions = []
for _, row in classics.iterrows():
    pred, _, probs = learners_v1[row.era].predict(PILImage.create(PROBE / row.filename))
    predictions.append(dict(predicted=pred, confidence=probs.max().item()))

classics = pd.concat([classics.reset_index(drop=True), pd.DataFrame(predictions)], axis=1)
classics["mismatch"] = classics.apply(lambda r: r.predicted not in r.true_genres.split(), axis=1)
classics[["title", "era", "edition_year", "true_genres", "predicted", "confidence", "mismatch"]]

In [ ]:
# Repackaging rate: share of classic covers that look like a genre the book isn't
classics.groupby("era").mismatch.mean()